In [1]:
# pip install transformers datasets peft accelerate evaluate

In [2]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)

In [3]:
from peft import LoraConfig, get_peft_model, TaskType

In [4]:
from datasets import load_dataset

In [5]:
datasets = load_dataset("csv", data_files="general_knowledge_qa.csv")

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
datasets

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 930
    })
})

In [8]:
dataset = datasets['train'].train_test_split(test_size=0.2, seed=42)
print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['test'])}")

Train: 744 | Eval: 186


In [9]:
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 744
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 186
    })
})

In [10]:
# ─────────────────────────────────────────
# 2. Load Model & Tokenizer
# ─────────────────────────────────────────

In [11]:
MODEL_NAME = "google/flan-t5-base"

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [15]:
# !pip uninstall torchao

In [16]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


In [18]:
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable param groups: {len(trainable)}")

Trainable param groups: 144


In [19]:
print(type(model))

<class 'peft.peft_model.PeftModelForSeq2SeqLM'>


In [20]:
# ─────────────────────────────────────────
# 4. Tokenize Dataset
# ─────────────────────────────────────────

In [22]:
MAX_INPUT  = 25
MAX_TARGET = 16

In [23]:
def preprocess(examples):
    inputs = ["answer the question: " + q for q in examples["question"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length",
        return_tensors=None
    )

    labels = tokenizer(
        examples["answer"],
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length",
        return_tensors=None
    )

    label_ids = labels["input_ids"]

    # Replace pad token id with -100 — loss ignores padding
    # label_ids = [
    #     [(token if token != tokenizer.pad_token_id else -100) for token in label]
    #     for label in label_ids
    # ]

    model_inputs["labels"] = label_ids

    return model_inputs

In [24]:
tokenized = dataset.map(preprocess, batched=True, remove_columns=["question", "answer"])

Map:   0%|          | 0/744 [00:00<?, ? examples/s]

Map:   0%|          | 0/186 [00:00<?, ? examples/s]

In [27]:
tokenized['train'][0]

{'input_ids': [1525,
  8,
  822,
  10,
  363,
  33,
  30872,
  7,
  58,
  3,
  9,
  61,
  1531,
  13,
  4345,
  7,
  3,
  115,
  61,
  1531,
  13,
  7466,
  8606,
  15,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [205, 5, 1531, 13, 4811, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [28]:
sample = tokenized["train"][0]
valid = [l for l in sample["labels"] if l != -100]
print(f"Valid label tokens: {len(valid)}")

Valid label tokens: 16


In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
peft_model = model.to(device)

In [30]:
print(dataset["train"][0])

{'question': 'What are constellations? a) Group of planets b) Group of galaxies c) Group of stars d) Group of meteors', 'answer': 'C. Group of stars'}


In [31]:
print(dataset["train"].column_names)

['question', 'answer']


In [32]:
sample = tokenized["train"][0]

print("Keys:", sample.keys())
print("Labels:", sample["labels"])

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Labels: [205, 5, 1531, 13, 4811, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [34]:
# ─────────────────────────────────────────
# 5. Training Arguments
# ─────────────────────────────────────────

training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-lora-qa",
    num_train_epochs=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    weight_decay=0.01,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,                                              # ✅ disable fp16
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none"
)

In [36]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    padding=True
)

In [37]:
trainer = Seq2SeqTrainer(
    model=model,                    # ✅ PEFT model, not base_model
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [38]:
print(type(trainer.model))

<class 'peft.peft_model.PeftModelForSeq2SeqLM'>


In [39]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,21.977148,10.845215
2,19.917041,9.021077
3,13.272998,6.201375
4,12.335278,5.790924
5,11.710034,5.592336
6,11.529468,5.477234
7,11.215063,5.424245
8,11.117041,5.380798
9,11.186426,5.342102
10,10.516235,5.321126


TrainOutput(global_step=940, training_loss=12.262208329870345, metrics={'train_runtime': 466.465, 'train_samples_per_second': 31.9, 'train_steps_per_second': 2.015, 'total_flos': 641879648501760.0, 'train_loss': 12.262208329870345, 'epoch': 20.0})

In [50]:
model.save_pretrained("./flan-new-t5-lora-adapter")
tokenizer.save_pretrained("./flan-new-t5-lora-adapter")
print("Adapter saved!")

Adapter saved!


In [55]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

In [56]:
base_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
model = PeftModel.from_pretrained(base_model, "./flan-new-t5-lora-adapter")
tokenizer = AutoTokenizer.from_pretrained("./flan-new-t5-lora-adapter")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [57]:
model.eval()

def ask(question):
    prompt = "answer the question: " + question
    inputs = tokenizer(prompt, return_tensors="pt", max_length=128, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,              # beam search = better answers
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(ask("Who invented the Computer?"))

Case


In [59]:
# !pip install rouge_score evaluate

In [60]:
import evaluate

rouge = evaluate.load("rouge")

# ─────────────────────────────────────────
# Run on entire eval set
# ─────────────────────────────────────────
model.eval()
predictions = []
references  = []

for sample in dataset["test"]:
    question = sample["question"]
    reference = sample["answer"]

    # Get model prediction
    predicted = ask(question)

    predictions.append(predicted)
    references.append(reference)
    print(f"Q  : {question}")
    print(f"Ref: {reference}")
    print(f"Pred:{predicted}")
    print("-" * 40)

# Compute ROUGE
results = rouge.compute(predictions=predictions, references=references)

print("\n===== ROUGE Scores =====")
print(f"ROUGE-1 : {results['rouge1']:.4f}")   # unigram overlap
print(f"ROUGE-2 : {results['rouge2']:.4f}")   # bigram overlap
print(f"ROUGE-L : {results['rougeL']:.4f}")   # longest common subsequence

Q  : How many months are there in a year?
Ref: 12
Pred:12
----------------------------------------
Q  : Roman Calendar is named as Julian Calendar. True or False?
Ref: TRUE
Pred:Fase
----------------------------------------
Q  : Ball, Bat, Stump, Hockey stick. True or False?
Ref: Hockey stick
Pred:True
----------------------------------------
Q  : Doctor, Surgeon, Nurse, Teacher
Ref: Teacher
Pred:Aias
----------------------------------------
Q  : What is the movement of Earth around the Sun resulting in change of seasons called?
Ref: Revolution
Pred:Sun
----------------------------------------
Q  :  What is the name of this famous painting?
Ref: Mona Lisa
Pred:Aease
----------------------------------------
Q  : Where did the game of Badminton originate from?
Ref: Pune
Pred:Sese
----------------------------------------
Q  : What is the name of the biggest rain forest in the world?
Ref: The Amazon
Pred:Fase
----------------------------------------
Q  : What is the person who compiles a d